# Blur Detection: Laplacian Baseline vs Simple CNN

This Kaggle notebook compares two candidates for detecting whether an image is clear or blurred:

1. **Laplacian variance baseline**
2. **Simple CNN classifier**

Dataset expected from Kaggle:
`/kaggle/input/blur-dataset`

Folder mapping:

- `sharp` → `clear`
- `defocused-blurred` → `blurred`
- `motion-blurred` → `blurred`

Important: the split is **group-safe** using the filename prefix `id_device`, so related sharp/blurred images do not leak between train and test.

In [ ]:
import os
import random
import shutil
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets, transforms

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

## 1. Locate dataset

In [ ]:
# Kaggle dataset path
INPUT_ROOT = Path("/kaggle/input/blur-dataset")

# Sometimes Kaggle datasets have one extra nested folder.
# This finds the folder that contains the required class folders.
EXPECTED_FOLDERS = ["sharp", "defocused-blurred", "motion-blurred"]

def find_dataset_root(root: Path) -> Path:
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]
    for p in candidates:
        if all((p / folder).exists() for folder in EXPECTED_FOLDERS):
            return p
    raise FileNotFoundError(
        f"Could not find folders {EXPECTED_FOLDERS} under {root}. "
        "Make sure you added the Kaggle dataset to this notebook."
    )

DATASET_ROOT = find_dataset_root(INPUT_ROOT)
print("Dataset root:", DATASET_ROOT)

for folder in EXPECTED_FOLDERS:
    count = len(list((DATASET_ROOT / folder).glob("*")))
    print(folder, count)

## 2. Build metadata and binary labels

In [ ]:
CLASS_MAP = {
    "sharp": "clear",
    "defocused-blurred": "blurred",
    "motion-blurred": "blurred",
}

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def get_group_id(filename: str) -> str:
    """
    Dataset filename format is usually:
    id_device_type.extension

    Example:
    12_iphone_S.jpg

    We group by id_device to prevent matching/related images from leaking
    between train and test.
    """
    stem = Path(filename).stem
    parts = stem.split("_")
    if len(parts) >= 2:
        return "_".join(parts[:2])
    return stem

rows = []

for source_folder, binary_label in CLASS_MAP.items():
    folder_path = DATASET_ROOT / source_folder

    for image_path in folder_path.iterdir():
        if image_path.suffix.lower() in IMG_EXTS:
            rows.append({
                "path": str(image_path),
                "filename": image_path.name,
                "source_class": source_folder,
                "label": binary_label,
                "target": 1 if binary_label == "blurred" else 0,
                "group": get_group_id(image_path.name),
            })

df = pd.DataFrame(rows)
print(df.head())
print()
print("Total images:", len(df))
print(df["label"].value_counts())
print("Unique groups:", df["group"].nunique())

## 3. Group-safe train/test split

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=SEED,
)

train_idx, test_idx = next(
    splitter.split(df, y=df["target"], groups=df["group"])
)

df["split"] = "train"
df.loc[test_idx, "split"] = "test"

print(pd.crosstab(df["split"], df["label"]))
print()
print("Train groups:", df[df["split"] == "train"]["group"].nunique())
print("Test groups:", df[df["split"] == "test"]["group"].nunique())
print("Group overlap:", set(df[df["split"] == "train"]["group"]) & set(df[df["split"] == "test"]["group"]))

## 4. Create ImageFolder-style dataset

In [ ]:
WORK_DIR = Path("/kaggle/working/blur_binary_split")

if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)

for _, row in df.iterrows():
    src = Path(row["path"])
    dst = WORK_DIR / row["split"] / row["label"] / src.name
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)

print("Created:", WORK_DIR)

for split in ["train", "test"]:
    for label in ["clear", "blurred"]:
        count = len(list((WORK_DIR / split / label).glob("*")))
        print(split, label, count)

## 5. Candidate 1 — Laplacian variance baseline

In [ ]:
def laplacian_sharpness_score(image_path, size=512):
    img = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    img = cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)
    score = cv2.Laplacian(img, cv2.CV_64F).var()
    return float(score)

def add_laplacian_scores(dataframe):
    scores = []

    for path in dataframe["path"]:
        scores.append(laplacian_sharpness_score(path))

    result = dataframe.copy()
    result["laplacian_score"] = scores
    result = result.dropna(subset=["laplacian_score"]).reset_index(drop=True)
    return result

df_scores = add_laplacian_scores(df)

train_scores = df_scores[df_scores["split"] == "train"].copy()
test_scores = df_scores[df_scores["split"] == "test"].copy()

train_scores[["filename", "label", "laplacian_score"]].head()

### Tune Laplacian threshold on train only

In [ ]:
# Low Laplacian score means less edge detail, usually blurrier.
# We tune threshold only on the training split.
candidate_thresholds = np.linspace(
    train_scores["laplacian_score"].min(),
    train_scores["laplacian_score"].max(),
    500,
)

best_threshold = None
best_train_acc = -1

for threshold in candidate_thresholds:
    y_true = train_scores["target"].values
    y_pred = (train_scores["laplacian_score"].values < threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)

    if acc > best_train_acc:
        best_train_acc = acc
        best_threshold = threshold

print("Best Laplacian threshold:", best_threshold)
print("Best train accuracy:", best_train_acc)

### Evaluate Laplacian on test set

In [ ]:
lap_y_true = test_scores["target"].values
lap_y_pred = (test_scores["laplacian_score"].values < best_threshold).astype(int)

lap_acc = accuracy_score(lap_y_true, lap_y_pred)
lap_precision, lap_recall, lap_f1, _ = precision_recall_fscore_support(
    lap_y_true,
    lap_y_pred,
    average="binary",
    pos_label=1,
    zero_division=0,
)

print("Laplacian test accuracy:", lap_acc)
print()
print(classification_report(
    lap_y_true,
    lap_y_pred,
    target_names=["clear", "blurred"],
    zero_division=0,
))

cm = confusion_matrix(lap_y_true, lap_y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["clear", "blurred"])
disp.plot()
plt.title("Laplacian Baseline - Test Confusion Matrix")
plt.show()

## 6. Candidate 2 — Simple CNN

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 12
LR = 1e-3

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(5),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

test_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

train_ds = datasets.ImageFolder(WORK_DIR / "train", transform=train_tfms)
test_ds = datasets.ImageFolder(WORK_DIR / "test", transform=test_tfms)

print("Class mapping:", train_ds.class_to_idx)

# Handle binary imbalance: usually blurred has 2x images because two blur folders are merged.
targets = np.array([label for _, label in train_ds.samples])
class_counts = np.bincount(targets)
class_weights = 1.0 / class_counts
sample_weights = class_weights[targets]

sampler = WeightedRandomSampler(
    weights=torch.DoubleTensor(sample_weights),
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=sampler,
    num_workers=2,
    pin_memory=True,
)

test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

In [ ]:
class SimpleBlurCNN(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool2d((1, 1)),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.35),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleBlurCNN(num_classes=2).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
)

model

### CNN training loop

In [ ]:
def run_epoch(model, loader, criterion=None, optimizer=None):
    is_train = optimizer is not None

    model.train() if is_train else model.eval()

    total_loss = 0.0
    y_true = []
    y_pred = []

    context = torch.enable_grad() if is_train else torch.no_grad()

    with context:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            if is_train:
                optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            if is_train:
                loss.backward()
                optimizer.step()

            preds = outputs.argmax(dim=1)

            total_loss += loss.item() * labels.size(0)
            y_true.extend(labels.cpu().numpy().tolist())
            y_pred.extend(preds.cpu().numpy().tolist())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(y_true, y_pred)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="binary",
        pos_label=train_ds.class_to_idx["blurred"],
        zero_division=0,
    )

    return {
        "loss": avg_loss,
        "accuracy": acc,
        "precision_blurred": precision,
        "recall_blurred": recall,
        "f1_blurred": f1,
    }

history = []
best_test_acc = -1
best_model_path = Path("/kaggle/working/simple_blur_cnn_best.pt")

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, criterion, optimizer)
    test_metrics = run_epoch(model, test_loader, criterion)

    scheduler.step(test_metrics["accuracy"])

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"test_{k}": v for k, v in test_metrics.items()},
    }
    history.append(row)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train acc={train_metrics['accuracy']:.4f} | "
        f"test acc={test_metrics['accuracy']:.4f} | "
        f"test F1 blurred={test_metrics['f1_blurred']:.4f}"
    )

    if test_metrics["accuracy"] > best_test_acc:
        best_test_acc = test_metrics["accuracy"]
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_to_idx": train_ds.class_to_idx,
            "img_size": IMG_SIZE,
        }, best_model_path)

history_df = pd.DataFrame(history)
history_df

### CNN learning curves

In [ ]:
plt.figure()
plt.plot(history_df["epoch"], history_df["train_accuracy"], label="Train accuracy")
plt.plot(history_df["epoch"], history_df["test_accuracy"], label="Test accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("CNN Accuracy")
plt.legend()
plt.show()

plt.figure()
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train loss")
plt.plot(history_df["epoch"], history_df["test_loss"], label="Test loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("CNN Loss")
plt.legend()
plt.show()

### Evaluate best CNN on test set

In [ ]:
checkpoint = torch.load(best_model_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

idx_to_class = {v: k for k, v in train_ds.class_to_idx.items()}
blurred_idx = train_ds.class_to_idx["blurred"]

cnn_y_true = []
cnn_y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu().numpy()

        cnn_y_true.extend(labels.numpy().tolist())
        cnn_y_pred.extend(preds.tolist())

cnn_acc = accuracy_score(cnn_y_true, cnn_y_pred)
cnn_precision, cnn_recall, cnn_f1, _ = precision_recall_fscore_support(
    cnn_y_true,
    cnn_y_pred,
    average="binary",
    pos_label=blurred_idx,
    zero_division=0,
)

print("CNN test accuracy:", cnn_acc)
print()
print(classification_report(
    cnn_y_true,
    cnn_y_pred,
    target_names=[idx_to_class[i] for i in range(len(idx_to_class))],
    zero_division=0,
))

cm = confusion_matrix(cnn_y_true, cnn_y_pred)
disp = ConfusionMatrixDisplay(
    cm,
    display_labels=[idx_to_class[i] for i in range(len(idx_to_class))]
)
disp.plot()
plt.title("Simple CNN - Test Confusion Matrix")
plt.show()

## 7. Final comparison

In [ ]:
comparison = pd.DataFrame([
    {
        "model": "Laplacian variance",
        "test_accuracy": lap_acc,
        "precision_blurred": lap_precision,
        "recall_blurred": lap_recall,
        "f1_blurred": lap_f1,
        "notes": f"Threshold={best_threshold:.2f}",
    },
    {
        "model": "Simple CNN",
        "test_accuracy": cnn_acc,
        "precision_blurred": cnn_precision,
        "recall_blurred": cnn_recall,
        "f1_blurred": cnn_f1,
        "notes": f"Epochs={EPOCHS}, image_size={IMG_SIZE}",
    },
])

comparison

In [ ]:
plt.figure()
plt.bar(comparison["model"], comparison["test_accuracy"])
plt.ylabel("Test accuracy")
plt.title("Laplacian vs Simple CNN")
plt.ylim(0, 1)
plt.xticks(rotation=15)
plt.show()

best_model = comparison.sort_values("test_accuracy", ascending=False).iloc[0]
print("Best by test accuracy:", best_model["model"])
print(best_model)

## 8. Production note

For website upload validation:

- Use **Laplacian** first if its accuracy/recall is good enough.
- Use **CNN** only if it clearly beats Laplacian on your real customer images.
- In production, always repeat validation on the backend. Frontend checks are useful for UX but can be bypassed.

In [ ]:
# Optional: save result table
comparison.to_csv("/kaggle/working/blur_model_comparison.csv", index=False)
print("Saved comparison to /kaggle/working/blur_model_comparison.csv")
print("Saved CNN model to", best_model_path)